In [2]:
# تثبيت مكتبة Transformers و Datasets من Hugging Face
# وتثبيت مكتبة Arabert المخصصة لمعالجة النصوص العربية
!pip install transformers[torch] datasets arabert farasa pyarabic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 16.7 MB/s eta 0:00:00
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186456 sha256=7107aa660a8f1d7afa7d1003b5d26462175c83defc34ba368f7ad27dfa40763f
  Stored in directory: /root/.cache/pip/wheels/bb/f1/26/f9002669ef6ad80a3c9f1b22880b35d9b4c6650011acee0523
Successfully built emoji


In [3]:
from datasets import Dataset
import pandas as pd

# إنشاء مجموعة بيانات مكثفة تغطي الفئات الأربع (سياسة، اقتصاد، رياضة، طقس)
news_data = {
    "text": [
        # --- اقتصاد ---
        "ارتفاع مؤشرات البورصة اليوم بفضل أسهم قطاع البنوك",
        "تراجع سعر صرف العملة المحلية أمام الدولار في التداولات الصباحية",
        "البنك المركزي يرفع أسعار الفائدة للحد من معدلات التضخم",
        "اتفاقية تجارية جديدة بين الصين والولايات المتحدة لتعزيز الصادرات",
        "انخفاض أسعار النفط عالمياً بعد زيادة المخزون الأمريكي",

        # --- رياضة ---
        "النادي الأهلي يتوج بلقب الدوري بعد فوزه في المباراة الأخيرة",
        "المنتخب الوطني يستعد لتصفيات كأس العالم بمعسكر مغلق",
        "إصابة نجم خط الوسط ستبعده عن الملاعب لمدة ثلاثة أسابيع",
        "انطلاق منافسات البطولة العربية للتنس في القاهرة بمشاركة واسعة",
        "نتائج قرعة دوري أبطال أوروبا تسفر عن مواجهات قوية",

        # --- سياسة ---
        "وزير الخارجية يلتقي بنظيره الفرنسي لمناقشة الأزمات الإقليمية",
        "البرلمان يصوت بالأغلبية على قانون الانتخابات الجديد",
        "قمة دولية في بروكسل لبحث ملف التغير المناخي والأمن الدولي",
        "الإعلان عن تشكيل حكومة ائتلافية جديدة لإنهاء الأزمة السياسية",
        "الرئيس يفتتح عدداً من المشاريع القومية في العاصمة",

        # --- طقس ---
        "تحذيرات من عاصفة رملية تضرب المناطق الصحراوية غداً",
        "هطول أمطار غزيرة تسبب في عرقلة حركة السير بوسط المدينة",
        "درجات الحرارة تتخطى الأربعين في موجة حر تستمر لثلاثة أيام",
        "الأرصاد الجوية تتوقع تساقط الثلوج على المرتفعات الجبلية",
        "استقرار في حالة الطقس وهدوء في سرعة الرياح على كافة الأنحاء"
    ],
    # التسميات (Labels): 0=سياسة، 1=اقتصاد، 2=رياضة، 3=طقس
    "label": [1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3]
}

# تحويل البيانات إلى تنسيق Datasets الخاص بـ Hugging Face
df = pd.DataFrame(news_data)
dataset = Dataset.from_pandas(df)

# تقسيم البيانات إلى تدريب واختبار (لضمان الدقة لاحقاً)
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(f"تم تجهيز {len(train_dataset)} جملة للتدريب و {len(test_dataset)} جملة للاختبار.")

تم تجهيز 16 جملة للتدريب و 4 جملة للاختبار.


In [4]:
from transformers import AutoTokenizer
from arabert.preprocess import ArabertPreprocessor

model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)
prep_object = ArabertPreprocessor(model_name=model_name)

def preprocess_function(examples):
    # تنظيف النص العربي أولاً باستخدام ArabertPreprocessor
    cleaned_texts = [prep_object.preprocess(t) for t in examples["text"]]
    # تحويل النصوص إلى أرقام (Tokens)
    return tokenizer(cleaned_texts, padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(preprocess_function, batched=True)
print("تمت عملية المعالجة والتقطيع بنجاح!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

تمت عملية المعالجة والتقطيع بنجاح!


In [5]:
from transformers import AutoModelForSequenceClassification

# تحميل نموذج BERT العربي مع تحديد عدد الفئات (4)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

print("تم تحميل نموذج AraBERT بنجاح.")

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

تم تحميل نموذج AraBERT بنجاح.


In [6]:
import torch

def predict(text):
    text_preprocessed = prep_object.preprocess(text)
    inputs = tokenizer(text_preprocessed, return_tensors="pt", padding=True, truncation=True, max_length=128)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    target_class = torch.argmax(probs).item()

    labels = ["سياسة", "اقتصاد", "رياضة", "طقس"]
    return labels[target_class]

# جرب كتابة جملة من عندك هنا
test_sentence = "توقعات بمنخفض جوي قادم من البحر المتوسط"
print(f"النص: {test_sentence}")
print(f"التصنيف المتوقع: {predict(test_sentence)}")

النص: توقعات بمنخفض جوي قادم من البحر المتوسط
التصنيف المتوقع: سياسة


In [7]:
from transformers import TrainingArguments, Trainer

# 1. إعدادات التدريب
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=20,
    per_device_train_batch_size=4,
    logging_steps=5,                # قللنا الرقم لنرى التحديثات بسرعة أكبر
    eval_strategy="epoch"
)

# 2. إنشاء المدرب (Trainer) مع تحديد الأقسام بدقة
trainer = Trainer(
    model=model,
    args=training_args,
    # هنا التعديل: حددنا قسم التدريب والاختبار من القاموس
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"]
)

# 3. ابدأ التدريب
print("بدء عملية التدريب... يرجى الانتظار")
trainer.train()
print("تم التدريب بنجاح!")

بدء عملية التدريب... يرجى الانتظار


Epoch,Training Loss,Validation Loss
1,No log,1.359643
2,1.244310,1.222985
3,1.208657,1.138019
4,0.717443,1.042132
5,0.594921,0.882705
6,0.594921,0.797175
7,0.336017,0.542318
8,0.179305,0.464708
9,0.122913,0.408417
10,0.078909,0.342789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

تم التدريب بنجاح!


In [8]:
import torch

def predict_final(text):
    # تحضير النص
    text_preprocessed = prep_object.preprocess(text)
    inputs = tokenizer(text_preprocessed, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # نقل البيانات للمعالج (GPU إذا كان متاحاً)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval() # وضع النموذج في نمط التقييم
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    target_class = torch.argmax(probs).item()

    labels = ["سياسة", "اقتصاد", "رياضة", "طقس"]
    return labels[target_class]

# اختبار حقيقي الآن
test_list = [
    "فوز ساحق للمنتخب في كأس آسيا",
    "انخفاض سعر صرف الريال مقابل العملات الأجنبية",
    "عاصفة ثلجية مرتقبة تضرب العاصمة"
]

for t in test_list:
    print(f"النص: {t} -> التصنيف: {predict_final(t)}")

النص: فوز ساحق للمنتخب في كأس آسيا -> التصنيف: رياضة
النص: انخفاض سعر صرف الريال مقابل العملات الأجنبية -> التصنيف: اقتصاد
النص: عاصفة ثلجية مرتقبة تضرب العاصمة -> التصنيف: طقس


In [9]:
import torch

def predict_news_category(text):
    # 1. تنظيف النص العربي بنفس الطريقة التي تدرب عليها النموذج
    text_preprocessed = prep_object.preprocess(text)

    # 2. تحويل النص إلى أرقام (Tokens) ونقلها إلى GPU
    inputs = tokenizer(text_preprocessed, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 3. وضع النموذج في نمط التوقع (Evaluation Mode)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # 4. استخراج الفئة الأعلى احتمالاً
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    target_class = torch.argmax(probs).item()

    # ترتيب التصنيفات كما وضعناها في الداتا
    labels = ["سياسة", "اقتصاد", "رياضة", "طقس"]
    return labels[target_class], probs[0][target_class].item()

# --- جرب النموذج هنا بجمل من اختيارك ---
test_sentences = [
    "انخفاض حاد في أسعار صرف العملات الرقمية هذا الصباح",
    "توقعات بهبوب رياح قوية وموجة غبار على المناطق الوسطى",
    "اجتماع وزراء الخارجية العرب لمناقشة ملف السلام",
    "تأهل الفريق إلى دور نصف النهائي بعد مباراة حماسية"
]

print("-" * 30)
for sentence in test_sentences:
    category, confidence = predict_news_category(sentence)
    print(f"النص: {sentence}")
    print(f"التصنيف: {category} (نسبة الثقة: {confidence*100:.2f}%)")
    print("-" * 30)

------------------------------
النص: انخفاض حاد في أسعار صرف العملات الرقمية هذا الصباح
التصنيف: اقتصاد (نسبة الثقة: 99.25%)
------------------------------
النص: توقعات بهبوب رياح قوية وموجة غبار على المناطق الوسطى
التصنيف: طقس (نسبة الثقة: 99.48%)
------------------------------
النص: اجتماع وزراء الخارجية العرب لمناقشة ملف السلام
التصنيف: سياسة (نسبة الثقة: 97.17%)
------------------------------
النص: تأهل الفريق إلى دور نصف النهائي بعد مباراة حماسية
التصنيف: رياضة (نسبة الثقة: 98.89%)
------------------------------
